# 01 - Exploração e qualidade dos dados
CRISP-NET, fase 2.4 *Verificar a qualidade dos dados* (Metodologia p.26).

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import networkx as nx
import pandas as pd
from src.carregamento import carregar_upfd
from src.cascata import montar_cascatas

vertices, arestas, arvores = carregar_upfd()
cascatas = montar_cascatas(vertices, arestas, arvores)
EXCLUIDAS = {"politifact14940": "aparece nas listas fake e real do FakeNewsNet"}
validas = {k: G for k, G in cascatas.items() if G.graph["noticia"] not in EXCLUIDAS}
print(len(cascatas), "cascatas carregadas;", len(validas), "após exclusões")

## 1. Toda cascata é uma árvore direcionada (arborescência)?

In [ ]:
print(sum(nx.is_arborescence(G) for G in cascatas.values()), "de", len(cascatas))

## 2. Vértices sem horário (fora a raiz)

In [ ]:
print(vertices.loc[~vertices["eh_raiz"], "horario"].isna().sum())

## 3. Algum filho compartilhou antes do pai?

In [ ]:
horario = vertices.set_index("vertice")["horario"]
pares = arestas.assign(h_pai=arestas["pai"].map(horario), h_filho=arestas["filho"].map(horario))
pares = pares.dropna(subset=["h_pai"])            # a raiz não tem horário, então as arestas que saem dela não entram
print("arestas entre usuários:", len(pares), "| filho antes do pai:", (pares["h_filho"] < pares["h_pai"]).sum())

## 4. Usuários repetidos na mesma cascata

In [ ]:
nao_raiz = vertices[~vertices["eh_raiz"]].copy()
repetido = nao_raiz.duplicated(subset=["arvore", "usuario"], keep="first")
print(f"{repetido.sum()} vértices ({repetido.mean():.1%}) em {nao_raiz.loc[repetido, 'arvore'].nunique()} cascatas")

## 5. Tamanho das cascatas

In [ ]:
tamanho = pd.Series({k: G.number_of_nodes() for k, G in cascatas.items()}, name="tamanho")
print(tamanho.describe().round(1)); print("< 10 vértices:", (tamanho < 10).sum())

## 6. Duração e concentração no tempo
As cascatas reais se estendem por anos; as falsas concentram-se na 1ª semana.

In [ ]:
inicio = nao_raiz.groupby("arvore")["horario"].transform("min")
nao_raiz["horas"] = (nao_raiz["horario"] - inicio).dt.total_seconds() / 3600
por_arvore = nao_raiz.groupby("arvore")["horas"].agg(
    duracao="max", nas_24h=lambda h: (h <= 24).mean(), nos_7dias=lambda h: (h <= 168).mean(),
).join(arvores.set_index("arvore")["rotulo"])
por_arvore.groupby("rotulo").median().round(2)

## 7. Exclusões

In [ ]:
print(arvores[arvores["noticia"].isin(EXCLUIDAS)])
arvores[~arvores["noticia"].isin(EXCLUIDAS)]["rotulo"].value_counts()